In [13]:
import polars as pl
from torch_geometric.data import OnDiskDataset

test_df = pl.read_parquet("../../data/test/triad/staged/test_triad.neg.parquet")

test_dset = OnDiskDataset("../../data/test/triad/graphs/test_triad_af3")

In [25]:
from torch_geometric.utils import to_dense_adj, get_laplacian
import torch

y_true = []
y_score = []

for graph in test_dset:
    lap = get_laplacian(graph.edge_index, )
    # edge_weight=graph.edge_attr[:, 0])

    adj_lap = to_dense_adj(lap[0], edge_attr=lap[1])

    eigvals, eigvecs = torch.linalg.eigh(adj_lap)

    y_true.append(graph.y[0])
    y_score.append(torch.topk(eigvals[0], 2, largest=False).values[1])

In [26]:
adj_lap

tensor([[[13., -1., -1.,  ...,  0.,  0.,  0.],
         [-1., 13., -1.,  ...,  0.,  0.,  0.],
         [-1., -1.,  8.,  ...,  0.,  0.,  0.],
         ...,
         [ 0.,  0.,  0.,  ..., 10., -1., -1.],
         [ 0.,  0.,  0.,  ..., -1.,  7., -1.],
         [ 0.,  0.,  0.,  ..., -1., -1.,  6.]]])

In [27]:
from sklearn.metrics import roc_auc_score

roc_auc_score(y_true, y_score)

0.36111111111111116

In [12]:
test_dset.close()